# 02 — Stage 1 Planner fine-tune

QLoRA fine-tune of `Qwen2.5-7B-Instruct` on `(english -> pseudocode)` pairs
from `data/stage1_planner/englishtopseudo.jsonl`.

Registers the DSL special tokens (`<PLAN>`, `</PLAN>`, `<STEP>`) from
`src/dsl/schema.py` into the tokenizer before training, so the model can
emit them as single tokens rather than spelling them out character by
character.

See `docs/colab_setup.md` (in this repo) for GPU choice and how to get the
data/schema files into this Colab runtime before running the cells below.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl

## 2. Config

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "pythonllm/data/stage1_planner/englishtopseudo.jsonl"
SCHEMA_PATH = "pythonllm/src/dsl/schema.py"
OUTPUT_DIR = "stage1_planner_qlora"

MAX_SEQ_LEN = 512
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

## 3. Load tokenizer and register DSL special tokens

Imports `SPECIAL_TOKENS` straight from `src/dsl/schema.py` so the tokenizer
always matches the DSL definition in the repo, instead of hardcoding the
token strings here.

In [ ]:
import sys

sys.path.insert(0, "pythonllm/src")
from dsl.schema import SPECIAL_TOKENS

print("DSL special tokens:", SPECIAL_TOKENS)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": SPECIAL_TOKENS}
)
print(f"Added {num_added} new special tokens; vocab size now {len(tokenizer)}")

## 4. Load base model in 4-bit and resize embeddings

The tokenizer grew by 3 tokens above, so the model's embedding matrix must
be resized to match before training touches it.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False

## 5. LoRA setup

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Load and format the dataset

Each example is formatted with the base model's own chat template so the
planner learns to respond to an instruction-style prompt with a pseudocode
plan, then loss-masked so only the pseudocode completion (not the prompt)
contributes to the loss.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw_dataset)
print(raw_dataset[0])

In [ ]:
SYSTEM_PROMPT = (
    "You are a planner that turns a task description into a pseudocode plan "
    "using the pythonllm DSL. Respond with only the plan, wrapped in "
    "<PLAN>...</PLAN> and made of <STEP> lines."
)


def format_example(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["english"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = prompt_text + example["pseudocode"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    labels = list(full["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    full["labels"] = labels
    return full


tokenized_dataset = raw_dataset.map(
    format_example, remove_columns=raw_dataset.column_names
)

## 7. Train

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

## 8. Save the adapter and tokenizer

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved LoRA adapter + tokenizer to {FINAL_DIR}")

## 9. Copy the adapter to Google Drive

Colab runtimes are ephemeral — copy the trained adapter out before the
session ends or disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/pythonllm_checkpoints
!cp -r {FINAL_DIR} /content/drive/MyDrive/pythonllm_checkpoints/stage1_planner

## 10. Quick sanity check

Run one held-in example through the fine-tuned model to confirm it emits a
well-formed `<PLAN>...</PLAN>` and that the special tokens decode as single
tokens rather than being spelled out.

In [ ]:
model.eval()
test_english = raw_dataset[0]["english"]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_english},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        inputs, max_new_tokens=MAX_SEQ_LEN, do_sample=False
    )

print("INPUT:", test_english)
print("OUTPUT:", tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=False))